In [1]:
import pandas as pd
import ollama
#model = "llama4:latest"
#model = "llama3.2:latest"
model = "llama3.1:8b"
#model = "deepseek-coder-v2:latest"
#model = "mistral:latest"

In [2]:
# Define the inclusion criteria
prompt = """
Your response MUST be a single comma-separated line with exactly 11 fields, in lowercase, without explanatory text. Commas are only allowed to separate fields—do not use them inside any field value:
  classification (1 or 0), species, disease, country, statistics, optimization, emphasis, strategies, objective_function, variables, constraints
  If not specified or unclear, write 'unknown' for every field.  

  INSTRUCTIONS:
As a systematic review expert, evaluate every provided article title {Title} and abstract {Abstract} against the specified inclusion criteria.

1. Carefully read the article title and abstract.

2. Inclusion criteria (Two criteria must be met for inclusion):
  a. Modelling: The study must involved quantitative statistical modelling, mathematical simulations or computational methods to analyze disease dynamics, epidemic model or surveillance data.
  b. Optimization: The study must apply optimization methods (e.g., multi-objective optimization, optimal control, reinforcement learning, algorithmic enhancements or others, not qualitative) to improve efficiency, decision-making, or predictive accuracy in disease surveillance.
   - Only include studies that provide explicit evidence of both quantitative modeling and the actual application of optimization methods. If modeling or optimization is only mentioned, unclear, or not clearly demonstrated in the abstract or title, classify as 0 (not included).
   - If the study only mentions optimization without applying it, classify as 0 (not included).
  Optimization is: the process of finding the best solution to a problem by maximizing or minimizing an objective function under a set of constraints.

3. ONLY use one of these two responses in the classification field:
  - If inclusion criteria is met: 1
  - If not, or if there is any doubt or missing information: 0
  - Never invent or generalize information. If the information is not present, write 'unknown'.
IMPORTANT: 
Only classify as 1, if BOTH quantitative modeling AND optimization methods are CLEARLY and EXPLICITLY APPLIED (not just mentioned) in the abstract or title. If EITHER is only mentioned, unclear, or missing, classify as 0 (not included). When in doubt, always classify as 0.

  - Do NOT use ';' or '/', newlines, or any other format.
  - Do not use any additional text or explanations outside the specified 11 fields.
 
4.  Exclude (classify as 0) studies that do not meet the inclusion criteria, such as:
    - Literature reviews, meta-analyses, laboratory techniques, cell culture, PCR or ELISA, molecular techniques, microbiological techniques, field trials, in vivo, ex vivo, in vitro.

EXTRACTION RULES:
5. Extract only information that is explicitly stated in the title or abstract. Do not infer, guess, or invent any information. If a field is not clearly mentioned, write 'unknown'.
  a. species: Use common names in singular form; if only Latin names exist, use them. Do not list diseases as species. If multiple species, separate by dash '-'. (e.g., pig, cattle, poultry, wild boar, dog, human, mice, sheep). Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  b. disease: If the study focuses on a specific disease, include it here. (e.g., african swine fever, foot and mouth disease, brucellosis, HPAI). Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  c. country: Specify the country or countries where the study was conducted. If multiple countries are involved, list them all, separated by dash '-'. Only include recognized countries do not include regions or continents. Use worldwide if the study covers all countries. Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  d. statistics: Extract the main statistical or modelling methods applied in the study using standardized terms only (e.g., regression analysis, probabilistic methods, mechanistic models, bayesian inference, compartment models, machine learning). Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  e. optimization: Name the reported optimization method (e.g., optimal control, direct optimal control, reinforcement learning, multi objective optimization, algorithmic enhancements, etc). Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
      Only include methods from optimization theory or mathematical optimization. Do not include general uses of the word "optimize" or non-technical meanings. Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
      If optimization is only mentioned but not implemented or actively applied, write mentioned but not applied. If not specified, write 'unknown'. Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  f. emphasis: Use only this values: surveillance, control or unknown. Do not use any other value. Write 'surveillance' if mentions are made to find/detect new cases of disease. Write 'control' if mentioned activities to control diseases as eradication, vaccination, culling, quarantine. Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  g. strategies: Extract the specific recommended strategies linked with surveillance or control (e.g., risk-based-monitoring, early-warning, vaccination, carcass-removal, culling, surveillance). If multiple strategies, separate by dash '-'. Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  h. objective_function: if classification = 1 look for the  objective function, write it here. (e.g., cost, profit, error, loss function). Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  i. variables: if classification = 1 look for the variables: These are the inputs or decision parameters that can be adjusted to optimize the objective function. Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.
  j. constraints: if classification = 1 look for the constraints: These are conditions that the variables must satisfy, such as equality or inequality constraints. Only extract what is explicitly mentioned. Do not use synonyms, generalizations, or invented terms.

- If multiple values for species, country, statistics, optimization, emphasis, strategies, variables or constraints separate by dash '-'. For all other fields, only one value is allowed.

- If any field is not specified or unclear, write 'unknown' for that field.
Examples:
  1, pig, african swine fever, germany, deterministic modelling, optimal control, surveillance, carcass testing, cost, pig density - culling rate, budget  
  1, poultry, avian influenza, france, modelling, cost function, control, vaccination, cost, vaccination rate, personal
  0, dog, rabies, vietnam, regression analysis, mentioned but not applied, risk factors, unknown, optimization only mentioned, unknown, unknown, unknown
  0, cattle, unknown, usa, qualitative analysis, mentioned but not applied, surveillance, unknown, optimization only mentioned, unknown, unknown, unknown
  0, wildboar, classical swine fever, canada, fixed effects, unknown, unknown, optimization only mentioned, unknown, unknown

Your task is to provide a clear, binary assessment (1 or 0) and extract the information as described above, single line format, no comments or notes."""

In [3]:
def classify_row(row):
    prompt_text = f"{prompt}\n\nTitle: {row['Title']}\nAbstract: {row['Abstract']}"
    response = ollama.generate(
        model=model,
        prompt=prompt_text,
    )
    # Expecting output like: true, dog, usa, logistic regression, 
    parts = [p.strip() for p in response['response'].strip().lower().split(',')]
    # Ensure we always have 11 parts
    while len(parts) < 11:
        parts.append('unknown')
    classification, species, disease, country, statistics, optimization, emphasis, strategies, objective_function, variables, constraints = parts[:11]
    return pd.Series({
        'classification': '1' if classification == '1' else '0',
        'species': species,
        'disease': disease,
        'country': country,
        'statistics': statistics,
        'optimization': optimization,
        'emphasis': emphasis,
        'strategies': strategies,
        'objective_function': objective_function,
        'variables': variables,
        'constraints': constraints
    })

In [6]:
# Load the CSV file
#df = pd.read_csv('pubmed_articles_general.csv', delimiter=',')
df = pd.read_csv('lr.csv', delimiter=',')

# Apply the classification function to each row and expand the results into new columns
df[['classification', 'species',  'disease', 'country', 'statistics', 'optimization', 'emphasis', 'strategies', 'objective_function', 'variables', 'constraints']] = df.apply(classify_row, axis=1)

# Save the updated dataframe
#df.to_csv('classified_papers_general.csv', index=False)
df.to_csv('classified_papers_lr.csv', index=False)

In [22]:
df = pd.read_csv('classified_papers_lr.csv', delimiter=',', keep_default_na=True)
df

,PMID,Item Type,Publication Year,Author,Title,Publication Title,ISBN,ISSN,DOI,Url,...,species,disease,country,statistics,optimization,emphasis,strategies,objective_function,variables,constraints
0,2WRZ3LCF,journalArticle,2020,"Knific, Tanja; Ocepek, Matjaž; Kirbiš, Andrej;...",Implications of Cattle Trade for the Spread an...,Frontiers in Veterinary Science,NaN,2297-1769,10.3389/fvets.2019.00454,https://www.frontiersin.org/article/10.3389/fv...,...,cattle,mycobacterium avium subsp. paratuberculosis,slovenia,static network analysis - si model,unknown,surveillance,targeted removal - collection centers - mounta...,unknown,inequality constraints,network connectedness - transmission probability
1,VRL5S9WK,journalArticle,2020,"Cheng, Q; Collender, PA; Heaney, AK; Li, XT; D...",The DIOS framework for optimizing infectious d...,PLOS COMPUTATIONAL BIOLOGY,NaN,1553-734X,10.1371/journal.pcbi.1008477,NaN,...,pig-dog-human,unknown,worldwide,mathematical optimization - numerical methods,multi-objective optimization,control,optimization only mentioned,cost-function,unknown,number of surveillance sites - target populations
2,C5NNIMFK,journalArticle,2019,"Chowdhury, S; Marufuzzaman, M; Tunc, H; Bian, ...",A modified Ant Colony Optimization algorithm t...,JOURNAL OF COMPUTATIONAL DESIGN AND ENGINEERING,NaN,2288-5048,10.1016/j.jcde.2018.10.004,NaN,...,unknown,unknown,unknown,unknown,optimization,surveillance,alns based immigrant schemes - aco,unknown,unknown,unknown
3,WGEZZU8P,journalArticle,2024,"Thiruvenkatam, P; Thangavel, P; Balasubramania...",Video Surveillance System-Based Human Activity...,INTERNATIONAL JOURNAL OF PATTERN RECOGNITION A...,NaN,0218-0014,10.1142/S021800142456007X,NaN,...,unknown,unknown,unknown,machine learning,garra rufa fish optimization,surveillance,unknown,cost - accuracy rate,features - parameters,equality constraints
4,UPD6BA3C,journalArticle,2018,"Biggs, A","Streptococcus uberis: environmental, contagiou...",CATTLE PRACTICE,NaN,0969-1251,NaN,NaN,...,unknown,streptococcus uberis,uk,unknown,mentioned but not applied,unknown,unknown,cost,herd size,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254,RGFJ3EXL,journalArticle,2022,"Shen, JDY; Mastrodicasa, D; Al Bulushi, Y; Lin...",Thoracic Endovascular Aortic Repair for Chroni...,RADIOGRAPHICS,NaN,0271-5333,10.1148/rg.220028,NaN,...,unknown,chronic type b aortic dissection,usa,imaging analysis,optimization not mentioned,surveillance,unknown,unknown,unknown,unknown
255,AJNUMSPZ,journalArticle,2023,"Srivastava, A; Prakash, J","Techniques, Answers, and Real-World UAV Implem...",WIRELESS PERSONAL COMMUNICATIONS,NaN,0929-6212,10.1007/s11277-023-10577-z,NaN,...,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
256,MTRSB2I7,journalArticle,2024,"Dammann, E; Ording-Müller, LS; Franchi-Abella,...",European Society of Pediatric Radiology survey...,PEDIATRIC RADIOLOGY,NaN,0301-0449,10.1007/s00247-023-05842-z,NaN,...,unknown,unknown,europe,unknown,unknown,surveillance,unknown,unknown,unknown,unknown
257,I9INM6T6,journalArticle,2022,"Zwager, LW; Moons, LMG; Sarasqueta, AF; Lacle,...",Long-term oncological outcomes of endoscopic f...,BMC GASTROENTEROLOGY,NaN,1471-230X,10.1186/s12876-022-02591-5,NaN,...,unknown,unknown,unknown,unknown,unknown,surveillance,unknown,unknown,unknown,unknown


In [23]:
included = (df['classification'] == 1).sum()
included

np.int64(27)

In [24]:
# Print statistics about the classification
total = len(df)
included = (df['classification'] == 1).sum()
not_included = (df['classification'] == 0).sum()

print(f"Included articles classified: {total}")
print(f"Included: {included}")
print(f"Not included: {not_included}")

# Filter only the included articles
idf = df[df['classification'] == 1]

# Print a table with the species   
print("\n-species:")
print(idf['species'].value_counts().to_frame('count'))

# Print a table with the diseases
print("\n-diseases:")
print(idf['disease'].value_counts().to_frame('count'))

# Print a table with the number of articles per country
print("\n-country:")
print(idf['country'].value_counts().to_frame('count'))

# Print a table with the statistical analysisused
print("\n-statistics:")
print(idf['statistics'].value_counts().to_frame('count'))

# Print a table with the optimization strategies used
print("\n-optimization:")
print(idf['optimization'].value_counts().to_frame('count'))

# Print a table with the objectives   
print("\n:-emphasis:")
print(idf['emphasis'].value_counts().to_frame('count'))

# Print a table with the strategies   
print("\n-strategies:")
print(idf['strategies'].value_counts().to_frame('count'))

# Print a table with the objective functions
print("\n-objective_function:")
print(idf['objective_function'].value_counts().to_frame('count'))

# Print a table with the variables
print("\n-variables:")
print(idf['variables'].value_counts().to_frame('count'))

# Print a table with the constraints
print("\n-constraints:")
print(idf['constraints'].value_counts().to_frame('count'))

Included articles classified: 259
Included: 27
Not included: 232

-species:
                                 count
species                               
unknown                             14
human                                2
pig-dog-human                        1
poultry                              1
human - avian influenza a virus      1
possum                               1
mycoplasma bovis                     1
clostridium perfringens              1
tick                                 1
hand foot and mouth disease          1
disease incidences                   1
influenza                            1
xylella fastidiosa                   1

-diseases:
                                                    count
disease                                                  
unknown                                                 7
zoonotic pathogens                                      1
influenza a viruses                                     1
schistosomiasis                      

In [25]:
# Print statistics about the general extraction
total = len(df)
included = (df['classification'] == 1).sum()
not_included = (df['classification'] == 0).sum()

print(f"Total articles classified: {total}")
print(f"Included: {included}")
print(f"Not included: {not_included}")

# - Species
print("\n -Species:")
print(df['species'].value_counts())
#print(df['species'].value_counts().head(20))
# - Disease
print("\n -Disease:")
print(df['disease'].value_counts())
#print(df['disease'].value_counts().head(20))
# - Country
print("\n -Country:")
print(df['country'].value_counts())
#print(df['country'].value_counts().head(20))
# - Statistics
print("\n -Statistics:")
print(df['statistics'].value_counts())
#print(df['statistics'].value_counts().head(20))
# - Optimization
print("\n -Optimization:")
print(df['optimization'].value_counts())
# - Emphasis
print("\n -Emphasis:")
print(df['emphasis'].value_counts())
# - Strategies
print("\n -Strategies:")
print(df['strategies'].value_counts())
# - Objective Function
print("\n -Objective Function:")
print(df['objective_function'].value_counts())
# - Variables
print("\n -Variables:")
print(df['variables'].value_counts())
# - Constraints
print("\n -Constraints:")
print(df['constraints'].value_counts())

Total articles classified: 259
Included: 27
Not included: 232

 -Species:
species
unknown                                           214
human                                               6
avian influenza                                     3
cattle                                              3
poultry                                             2
pig-dog-human                                       1
mallard                                             1
klebsiella pneumoniae - pseudomonas aeruginosa      1
avian influenza virus                               1
goose                                               1
finfish                                             1
human - avian influenza a virus                     1
bovine                                              1
horse                                               1
dengue fever                                        1
possum                                              1
mycoplasma bovis                                    1


In [26]:
# Print the first 10 included articles: only Title and classification response variables
print(idf[['Title', 'classification', 'statistics']].head(10))

                                                Title  classification  \
1   The DIOS framework for optimizing infectious d...               1   
10  Three-Way k-Means Model: Dynamic Optimal Senso...               1   
12  Fast Pig Detection with a Top-View Camera unde...               1   
13  Optimization of a Novel Non-invasive Oral Samp...               1   
24  Wastewater monitoring of human and avian influ...               1   
38  Optimized strategy for schistosomiasis elimina...               1   
44  The optimisation of Salmonella surveillance pr...               1   
50  Serological surveillance reveals patterns of e...               1   
51  Optimising cost-effectiveness of freedom from ...               1   
56  Optimization and evaluation of a non-invasive ...               1   

                                           statistics  
1       mathematical optimization - numerical methods  
10                             deterministic modeling  
12                        im

In [30]:
#human_df = pd.read_csv('human_classified_general.csv', delimiter=';', keep_default_na=True)
human_df = pd.read_csv('human_classified_papers_lr.csv', delimiter=';', keep_default_na=True)

print(human_df.columns.tolist())  # Check actual column names, be carefull
human_df
# Count NA values in the 'human' column
na_count = human_df['human'].isna().sum()
print(f"Number of NA values in 'human': {na_count}")

['PMID', 'Item Type', 'Publication Year', 'Author', 'Title', 'Publication Title', 'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract', 'Date', 'Date Added', 'Date Modified', 'Access Date', 'Pages', 'Num Pages', 'Issue', 'Volume', 'Number Of Volumes', 'Journal Abbreviation', 'Short Title', 'Series', 'Series Number', 'Series Text', 'Series Title', 'Publisher', 'Place', 'Language', 'Rights', 'Type', 'Archive', 'Archive Location', 'Library Catalog', 'Call Number', 'Extra', 'Notes', 'File Attachments', 'Link Attachments', 'Manual Tags', 'Automatic Tags', 'Editor', 'Series Editor', 'Translator', 'Contributor', 'Attorney Agent', 'Book Author', 'Cast Member', 'Commenter', 'Composer', 'Cosponsor', 'Counsel', 'Interviewer', 'Producer', 'Recipient', 'Reviewed Author', 'Scriptwriter', 'Words By', 'Guest', 'Number', 'Edition', 'Running Time', 'Scale', 'Medium', 'Artwork Size', 'Filing Date', 'Application Number', 'Assignee', 'Issuing Authority', 'Country', 'Meeting Name', 'Conference Name', 'Court', 'Referenc

In [32]:
# Comparing to human classified articles

# Load the human_classified file (make sure to use the correct delimiter it changes in my german regional config)
#human_df = pd.read_csv('human_classified_general.csv', delimiter=';', keep_default_na=True)
#df = pd.read_csv('classified_papers_general.csv', delimiter=',', keep_default_na=True)

human_df = pd.read_csv('human_classified_papers_lr.csv', delimiter=';', keep_default_na=True)
df = pd.read_csv('classified_papers_lr.csv', delimiter=',', keep_default_na=True)

human_df['PMID'] = human_df['PMID'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)


print(human_df.columns.tolist())  # Check actual column names, be carefull

# some colum checking
df['PMID'] = df['PMID'].astype(str).str.strip()
human_df['PMID'] = human_df['PMID'].astype(str).str.strip()

# Use the correct column name below!
#df['human'] = df['PMID'].isin(human_df.loc[human_df['human'] == 1, 'PMID']).astype('Int64')  # Use Int64 for nullable integer type

df = df.merge(human_df[['PMID', 'human']], on='PMID', how='left')
# Now df['human'] contains 1, 0, and NA exactly as in human_df

#print(human_df['human'].sum())  # Should be 7
print(df['human'].sum())  # Should be 7
print(df['classification'].sum())  # Should be 7

# Count the number of 0 values in the 'human' column
zero_count = (df['human'] == 0).sum()
print(f"Number of 0 values in 'human': {zero_count}")

print(df['human'].isna().sum())

['PMID', 'Item Type', 'Publication Year', 'Author', 'Title', 'Publication Title', 'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract', 'Date', 'Date Added', 'Date Modified', 'Access Date', 'Pages', 'Num Pages', 'Issue', 'Volume', 'Number Of Volumes', 'Journal Abbreviation', 'Short Title', 'Series', 'Series Number', 'Series Text', 'Series Title', 'Publisher', 'Place', 'Language', 'Rights', 'Type', 'Archive', 'Archive Location', 'Library Catalog', 'Call Number', 'Extra', 'Notes', 'File Attachments', 'Link Attachments', 'Manual Tags', 'Automatic Tags', 'Editor', 'Series Editor', 'Translator', 'Contributor', 'Attorney Agent', 'Book Author', 'Cast Member', 'Commenter', 'Composer', 'Cosponsor', 'Counsel', 'Interviewer', 'Producer', 'Recipient', 'Reviewed Author', 'Scriptwriter', 'Words By', 'Guest', 'Number', 'Edition', 'Running Time', 'Scale', 'Medium', 'Artwork Size', 'Filing Date', 'Application Number', 'Assignee', 'Issuing Authority', 'Country', 'Meeting Name', 'Conference Name', 'Court', 'Referenc

In [33]:
#%pip install statsmodels
# Check the se and sp calc by hand

import numpy as np
from statsmodels.stats.proportion import proportion_confint

# Map classification to binary: 1 for 'included', 0 for 'not included'
df['pred'] = (df['classification'] == 1).astype(int)
df['true'] = df['human'].astype('Int64')  # Use Int64 for nullable integer type
print(df['true'].isna().sum())
print((df['true'] == 1).sum())


# Sensitivity: proportion of true positives among all actual positives (human == 1)
tp = ((df['true'] == 1) & (df['pred'] == 1)).sum()
fn = ((df['true'] == 1) & (df['pred'] == 0)).sum()
tn = ((df['true'] == 0) & (df['pred'] == 0)).sum()
fp = ((df['true'] == 0) & (df['pred'] == 1)).sum()
total = len(df)
print(total)

# Metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
accuracy = (tp + tn) / total if total > 0 else np.nan

# 95% confidence intervals (using Wilson score interval)
sens_low, sens_upp = proportion_confint(tp, tp + fn, alpha=0.05, method='wilson')
spec_low, spec_upp = proportion_confint(tn, tn + fp, alpha=0.05, method='wilson')
acc_low, acc_upp = proportion_confint(tp + tn, total, alpha=0.05, method='wilson')

print(f"Sensitivity: {sensitivity:.3f} (95% CI: {sens_low:.3f}–{sens_upp:.3f})")
print(f"Specificity: {specificity:.3f} (95% CI: {spec_low:.3f}–{spec_upp:.3f})")
print(f"Accuracy:    {accuracy:.3f} (95% CI: {acc_low:.3f}–{acc_upp:.3f})")

# Save with metrics
df.to_csv('classified_papers_general_metric.csv', index=False, na_rep='')

0
14
259
Sensitivity: 0.571 (95% CI: 0.326–0.786)
Specificity: 0.922 (95% CI: 0.882–0.950)
Accuracy:    0.903 (95% CI: 0.861–0.934)


In [34]:
print(f' tp: {tp}')
print(f' fn: {fn}')
print(f' tn: {tn}')
print(f' fp: {fp}')
print(f"Number of positive samples in 'human': {human_df['human'].sum()}")  # Should be > 0 if you have positives
print(f"Number of positive predictions: {df['pred'].sum()}")   # Should be > 0 if the model included any

 tp: 8
 fn: 6
 tn: 226
 fp: 19
Number of positive samples in 'human': 14
Number of positive predictions: 27


In [35]:
mask = df['true'].notna()
tp = ((df['true'] == 1) & (df['pred'] == 1) & mask).sum()
fn = ((df['true'] == 1) & (df['pred'] == 0) & mask).sum()
tn = ((df['true'] == 0) & (df['pred'] == 0) & mask).sum()
fp = ((df['true'] == 0) & (df['pred'] == 1) & mask).sum()
total = mask.sum()  # Only rows with non-NA human

accuracy = (tp + tn) / total if total > 0 else np.nan

# 95% confidence intervals (using Wilson score interval)
sens_low, sens_upp = proportion_confint(tp, tp + fn, alpha=0.05, method='wilson')
spec_low, spec_upp = proportion_confint(tn, tn + fp, alpha=0.05, method='wilson')
acc_low, acc_upp = proportion_confint(tp + tn, total, alpha=0.05, method='wilson')

print(f"Sensitivity: {sensitivity:.3f} (95% CI: {sens_low:.3f}–{sens_upp:.3f})")
print(f"Specificity: {specificity:.3f} (95% CI: {spec_low:.3f}–{spec_upp:.3f})")
print(f"Accuracy:    {accuracy:.3f} (95% CI: {acc_low:.3f}–{acc_upp:.3f})")

print(f' tp: {tp}')
print(f' fn: {fn}')
print(f' tn: {tn}')
print(f' fp: {fp}')
print(f"Number of positive samples in 'human': {human_df['human'].sum()}")  # Should be > 0 if you have positives
print(f"Number of positive predictions: {df['pred'].sum()}")   # Should be > 0 if the model included any

# Count the number of 0 values in the 'human' column
zero_count = (df['human'] == 0).sum()
print(f"Number of 0 values in 'human': {zero_count}")

print(f"Number of NA values in 'human': {df['human'].isna().sum()}")

Sensitivity: 0.571 (95% CI: 0.326–0.786)
Specificity: 0.922 (95% CI: 0.882–0.950)
Accuracy:    0.903 (95% CI: 0.861–0.934)
 tp: 8
 fn: 6
 tn: 226
 fp: 19
Number of positive samples in 'human': 14
Number of positive predictions: 27
Number of 0 values in 'human': 245
Number of NA values in 'human': 0


In [ ]:
#%pip install geopandas matplotlib
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Load your classified data
#df = pd.read_csv('classified_papers.csv')

# Filter for included articles
included = df[df['classification'] == 'included']

# Get all countries (split if multiple countries per row)
countries = included['country'].dropna().str.split(' ')
countries = countries.explode().str.strip().str.title()  # Normalize country names

# Load world map
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Mark countries present in your data
world['included'] = world['name'].isin(countries)

# Plot
fig, ax = plt.subplots(figsize=(15, 8))
world.plot(ax=ax, color='lightgrey', edgecolor='white')
world[world['included']].plot(ax=ax, color='dodgerblue')
ax.set_title('Countries with Included Articles')
plt.axis('off')
plt.show()